In [ ]:
# Axolotl + bitsandbytes
#!pip -q install axolotl==0.4.0.post1 bitsandbytes==0.43.3
#!pip install --no-build-isolation axolotl[deepspeed] #T4
!pip install --upgrade --no-cache-dir git+https://github.com/OpenAccess-AI-Collective/axolotl.git@main#egg=axolotl[deepspeed]  # A100


DEPRECATION: git+https://github.com/OpenAccess-AI-Collective/axolotl.git@main#egg=axolotl[deepspeed] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Cloning https://github.com/OpenAccess-AI-Collective/axolotl.git (to revision main) to /tmp/pip-install-sp1jnqfq/axolotl_7b169f1de1ab4da7ae653eeffb5f9786
  Running command git clone --filter=blob:none --quiet https://github.com/OpenAccess-AI-Collective/axolotl.git /tmp/pip-install-sp1jnqfq/axolotl_7b169f1de1ab4da7ae653eeffb5f9786
  Resolved https://github.com/OpenAccess-AI-Collective/axolotl.git to commit 301e22849f41c67c31e065a222235ab120fd4074
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 267.3 MB/

#0) Mount + paths + helpers



In [ ]:
from pathlib import Path
import os, shutil, json

# ==== EDIT THESE IF YOUR DRIVE LAYOUT IS DIFFERENT ====
DRIVE_PROJECT = Path("/content/drive/MyDrive/ft_vs_rag_project")  # your project root on Drive
DRIVE_DATA_DIR = DRIVE_PROJECT / "data"                            # where you saved the split data & QA JSONLs
DRIVE_OUT_DIR  = DRIVE_PROJECT / "outputs"
#DRIVE_PREP_DATA_DIR = DRIVE_PROJECT / "notebook_outputs"                    # where we'll copy trained adapters & logs
#DRIVE_DATA_DIR = DRIVE_PROJECT / "ft_vs_rag_multidata/data_hotpot"                    # where we'll copy trained adapters & logs
# ======================================================

# Local workspace
LOCAL_ROOT     = Path("/content/ft_vs_rag_work")
LOCAL_DATA_DIR = LOCAL_ROOT / "data"
LOCAL_OUT_DIR  = LOCAL_ROOT / "outputs"
LOCAL_CFG_DIR  = LOCAL_ROOT / "axolotl_configs"
LOCAL_LOG_DIR  = LOCAL_ROOT / "logs"
for p in [LOCAL_DATA_DIR, LOCAL_OUT_DIR, LOCAL_CFG_DIR, LOCAL_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Mount Drive
from google.colab import drive  # type: ignore
drive.mount('/content/drive')

print("Drive data dir :", DRIVE_DATA_DIR)
#print("Drive prep data dir :", DRIVE_PREP_DATA_DIR) ###
print("Drive out  dir :", DRIVE_OUT_DIR)
print("Local root     :", LOCAL_ROOT)


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import zipfile
from typing import Optional, Iterable

def safe_copy_file(src: Path, dst: Path, overwrite: bool = True):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and not overwrite:
        return dst
    shutil.copy2(src, dst)
    return dst

def zip_dir(src_dir: Path, zip_path: Path):
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(src_dir):
            for fn in files:
                fp = Path(root) / fn
                zf.write(fp, arcname=str(fp.relative_to(src_dir)))
    return zip_path

def count_jsonl_rows(p: Path) -> int:
    n = 0
    with p.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1
    return n

def peek_jsonl(p: Path, k: int = 2):
    print(f"Peek {p.name}:")
    i = 0
    with p.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            keys = list(obj.keys())
            print({kk: obj.get(kk) for kk in keys[:5]})
            i += 1
            if i >= k: break

def find_in_drive(filename: str, base_dir: Path) -> Optional[Path]:
    for root, _, files in os.walk(base_dir):
        if filename in files:
            return Path(root) / filename
    return None


In [ ]:
DRIVE_PREP_DATA_DIR = DRIVE_PROJECT / "notebook_outputs"
DRIVE_PREP_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Copying files from {DRIVE_PREP_DATA_DIR} to {LOCAL_DATA_DIR}...")
for item in DRIVE_PREP_DATA_DIR.iterdir():
    if item.is_file():
        safe_copy_file(item, LOCAL_DATA_DIR / item.name)
        print(f"Copied {item.name}")
    elif item.is_dir():
        print(f"Skipping directory {item.name}")
print("Finished copying files.")

Copying files from /content/drive/MyDrive/ft_vs_rag_project/notebook_outputs to /content/ft_vs_rag_work/data...
Copied docs_qa_with_context_STABLEONLY.jsonl
Copied docs_qa_with_context.jsonl
Finished copying files.


In [ ]:
src_file = DRIVE_PROJECT / "ft_vs_rag_multidata" / "data_hotpot" / "stable_docs.jsonl"
dst_file = LOCAL_DATA_DIR / "stable_docs.jsonl"

if src_file.exists():
    safe_copy_file(src_file, dst_file)
    print(f"Copied {src_file.name} to {dst_file}")
else:
    print(f"Warning: Source file not found at {src_file}")

Copied stable_docs.jsonl to /content/ft_vs_rag_work/data/stable_docs.jsonl


# 1) Copy inputs Drive → Local (no writing to Drive during training)

In [ ]:
from pathlib import Path
import json, random

LOCAL_ROOT     = Path("/content/ft_vs_rag_work")
LOCAL_DATA_DIR = LOCAL_ROOT / "data"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# קבצי מקור אפשריים (התאימי לשמות שיש לך בפועל)
stable_docs = LOCAL_DATA_DIR / "stable_docs.jsonl"  # אמור להכיל טקסטים חופשיים {"text": "..."} או {"title": ..., "text": ...}
qa_stable   = LOCAL_DATA_DIR / "train_ft_qas.jsonl"
qa_all      = LOCAL_DATA_DIR / "docs_qa_with_context.jsonl"

# קבצי יעד
pretrain_out = LOCAL_DATA_DIR / "docs_for_pretrain.jsonl"
sft_out      = LOCAL_DATA_DIR / "docs_qa_with_context_ALPACA.jsonl"

def read_jsonl(p: Path):
    rows = []
    if not p.exists():
        return rows
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(p: Path, rows):
    with p.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ---------- Stage-1: מייצרים קובץ pretrain {"text": "..."} ----------
pre_rows = []
src = read_jsonl(stable_docs)
if src:
    for r in src:
        txt = r.get("text") or r.get("content") or ""
        if txt:
            pre_rows.append({"text": txt})
else:
    # fallback: אם אין stable_docs, ננסה לחלץ טקסטים מהקשרי ה-QA
    qa_src = read_jsonl(qa_stable) or read_jsonl(qa_all)
    for r in qa_src:
        ctx = r.get("context","")
        if ctx:
            pre_rows.append({"text": ctx})

# דוגם מעט אם גדול מאוד (כדי לא להעמיס בהתחלה)
MAX_PRETRAIN = 600_000
if len(pre_rows) > MAX_PRETRAIN:
    random.seed(0)
    pre_rows = random.sample(pre_rows, MAX_PRETRAIN)

write_jsonl(pretrain_out, pre_rows)
print(f"Wrote pretrain file: {pretrain_out} ({len(pre_rows)} rows)")

# ---------- Stage-2: ממיר QA+context לפורמט Alpaca ----------
system_msg = (
    "You are a helpful assistant. Use the provided context when it helps, "
    "but you are not required to rely on it exclusively. Respond with your best knowledge and reasoning."
)

qa_src = read_jsonl(qa_stable)
if not qa_src:
    qa_src = read_jsonl(qa_all)
assert qa_src, "Couldn't find QA source file (docs_qa_with_context_STABLEONLY.jsonl / docs_qa_with_context.jsonl)."

alpaca_rows = []
for r in qa_src:
    q = r.get("question","").strip()
    a = r.get("answer","").strip()
    ctx = r.get("context","").strip()
    # בפורמט alpaca: instruction + input + output
    # נשים את ה־system בהמשך בקונפיג (system_prompt), כאן נשמור instruction/input/output.
    instruction = "Answer the question. Use the provided context if it helps."
    inp = f"<CONTEXT>\n{ctx}\n\nQuestion: {q}" if ctx else f"Question: {q}"
    out = a
    row = {"instruction": instruction, "input": inp, "output": out}
    if "id" in r: row["id"] = r["id"]
    alpaca_rows.append(row)

write_jsonl(sft_out, alpaca_rows)
print(f"Wrote SFT (alpaca) file: {sft_out} ({len(alpaca_rows)} rows)")


Wrote pretrain file: /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl (533355 rows)
Wrote SFT (alpaca) file: /content/ft_vs_rag_work/data/docs_qa_with_context_ALPACA.jsonl (90447 rows)


#2) (Optional) Sequence length analysis (local files)

In [11]:
# Optional: keep if you use it to set SEQ_LEN
!pip -q install transformers>=4.44.0

from transformers import AutoTokenizer
import numpy as np

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tok.model_max_length = 8192

def jsonl_iter(path, limit=None):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def lengths_for_docs(jsonl_path, sample=3000):
    lens = []
    for row in jsonl_iter(jsonl_path, limit=sample):
        text = row.get("text","")
        ids = tok(text, add_special_tokens=True, truncation=False)["input_ids"]
        lens.append(len(ids))
    return np.array(lens, dtype=np.int32)

def lengths_for_qa(jsonl_path, sample=3000):
    lens = []
    sys_prefix = ""
    for row in jsonl_iter(jsonl_path, limit=sample):
        q = row.get("question","")
        a = row.get("answer","")
        ctx = row.get("context","")
        prompt = f"<|im_start|>system\n{sys_prefix}<|im_end|>\n" \
                 f"<|im_start|>user\nQuestion: {q}\nContext:\n{ctx}<|im_end|>\n" \
                 f"<|im_start|>assistant\n" + a
        ids = tok(prompt, add_special_tokens=True, truncation=False)["input_ids"]
        lens.append(len(ids))
    return np.array(lens, dtype=np.int32)
def lengths_for_qa(jsonl_path, sample=3000):
    lens = []
    sys_prefix = ""
    for row in jsonl_iter(jsonl_path, limit=sample):
        q = row.get("instruction", "")
        ctx = row.get("input", "")
        a = row.get("output", "")
        prompt = (
            f"<|im_start|>system\n{sys_prefix}<|im_end|>\n"
            f"<|im_start|>user\nInstruction: {q}\nInput:\n{ctx}<|im_end|>\n"
            f"<|im_start|>assistant\n{a}"
        )
        ids = tok(prompt, add_special_tokens=False)["input_ids"]
        lens.append(len(ids))
    return np.array(lens, dtype=np.int32)

docs_len = lengths_for_docs(LOCAL_DATA_DIR/"docs_for_pretrain.jsonl", sample=530000)
qa_len   = lengths_for_qa(LOCAL_DATA_DIR/"docs_qa_with_context_ALPACA.jsonl", sample=90000)

def summarize(name, arr):
    if arr.size == 0:
        print(f"{name}: no samples"); return
    for p in [50, 75, 90, 95, 99]:
        print(f"{name} P{p}: {int(np.percentile(arr, p))}")
    print(f"{name} max: {int(arr.max())}, mean: {arr.mean():.1f}, n={len(arr)}")

print("=== Sequence Length Summary ===")
summarize("docs", docs_len)
summarize("qa",   qa_len)

SEQ_LEN = int(min(4096, max(np.percentile(docs_len, 95), np.percentile(qa_len, 95))))
SEQ_LEN = (SEQ_LEN // 128) * 128
SEQ_LEN = max(2048, min(SEQ_LEN, 4096))
print("Chosen sequence_len:", SEQ_LEN)


=== Sequence Length Summary ===
docs P50: 159
docs P75: 316
docs P90: 721
docs P95: 758
docs P99: 767
docs max: 789, mean: 249.7, n=530000
qa P50: 610
qa P75: 1027
qa P90: 1509
qa P95: 1785
qa P99: 2341
qa max: 2504, mean: 773.3, n=90000
Chosen sequence_len: 2048


#3) Write Axolotl configs to local (separate caches + outputs)

In [ ]:
from pathlib import Path

LOCAL_ROOT    = Path("/content/ft_vs_rag_work")
LOCAL_CFG_DIR = LOCAL_ROOT / "axolotl_configs"
LOCAL_OUT_DIR = LOCAL_ROOT / "outputs"
LOCAL_DATA_DIR= LOCAL_ROOT / "data"
LOCAL_CFG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
SEQ_LEN    = 1760 #2048
STAGE1_DIR = (LOCAL_OUT_DIR / "mistral_stage1")
STAGE2_DIR = (LOCAL_OUT_DIR / "mistral_stage2")

pretrain_file = (LOCAL_DATA_DIR / "docs_for_pretrain.jsonl")
sft_file      = (LOCAL_DATA_DIR / "docs_qa_with_context_ALPACA.jsonl")

assert pretrain_file.exists(), f"missing {pretrain_file}"
assert sft_file.exists(),      f"missing {sft_file}"

STAGE1_MAX_STEPS = 2000  # אפשר לכייל

pretrain_yaml = f"""
base_model: {BASE_MODEL}
model_type: mistral
tokenizer_type: AutoTokenizer
trust_remote_code: true

###############
load_in_4bit: true
bnb_4bit_quant_type: nf4
bnb_4bit_compute_dtype: bfloat16
torch_dtype: bfloat16
gradient_checkpointing: true
#attn_implementation: flash_attention_2
attn_implementation: sdpa

sequence_len: {SEQ_LEN}
micro_batch_size: 8
gradient_accumulation_steps: 8
eval_micro_batch_size: 8

max_steps: {STAGE1_MAX_STEPS}

bf16: true
tf32: true

# DataLoader
# dataloader_num_workers: 0 #2
# dataloader_pin_memory: true

packing: true
pad_to_sequence_len: false

###########

#load_in_4bit: true
adapter: qlora
#bnb_4bit_quant_type: nf4
#bnb_4bit_compute_dtype: float16
#bnb_4bit_use_double_quant: true

# --- Stage 1 (pretrain) ---
dataset_prepared_path: cache_pretrain
debug_num_examples: 0

pretraining_dataset:
  - path: json
    data_files:
      - /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl
    text_column: text
    streaming: false
    shuffle: true
    seed: 42

output_dir: {STAGE1_DIR.as_posix()}

sequence_len: {SEQ_LEN}
#sample_packing: false
#pad_to_sequence_len: false

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
bias: "none"
task_type: "CAUSAL_LM"
lora_target_linear: true
adapters: lora

#gradient_checkpointing: true
#gradient_accumulation_steps: 4
#micro_batch_size: 1

num_epochs: 1
save_steps: 0
evals_per_epoch: 0

optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 2e-4
warmup_steps: 100
logging_steps: 10

# preprocessing_num_workers: 1
"""

sft_yaml = f"""
base_model: {BASE_MODEL}
model_type: mistral
tokenizer_type: AutoTokenizer
trust_remote_code: true

###############
load_in_4bit: true
bnb_4bit_quant_type: nf4
bnb_4bit_compute_dtype: bfloat16
torch_dtype: bfloat16
gradient_checkpointing: true
#attn_implementation: flash_attention_2
attn_implementation: sdpa

sequence_len: 2048
micro_batch_size: 8
gradient_accumulation_steps: 8
eval_micro_batch_size: 8

bf16: true
tf32: true

# DataLoader
# dataloader_num_workers: 0 #2
# dataloader_pin_memory: true

packing: true
pad_to_sequence_len: false

###########

#load_in_4bit: true
adapter: qlora
lora_path: {STAGE1_DIR.as_posix()}

#bnb_4bit_quant_type: nf4
#bnb_4bit_compute_dtype: float16
#bnb_4bit_use_double_quant: true

datasets:
  - path: {sft_file.as_posix()}
    type: alpaca
    prompt_template: alpaca
    streaming: false

system_prompt: |
  You are a helpful assistant. Use the provided context when it helps,
  but you are not required to rely on it exclusively. Respond with your best knowledge and reasoning.

dataset_prepared_path: cache_sft
output_dir: {STAGE2_DIR.as_posix()}

sequence_len: {SEQ_LEN}
#sample_packing: false
#pad_to_sequence_len: false

lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
bias: "none"
task_type: "CAUSAL_LM"
lora_target_linear: true
adapters: lora

#gradient_checkpointing: true
#gradient_accumulation_steps: 4
#micro_batch_size: 1
num_epochs: 1

optimizer: adamw_bnb_8bit
lr_scheduler: cosine
learning_rate: 1e-4
warmup_steps: 100
train_on_inputs: false
group_by_length: false
save_every: 0
evals_per_epoch: 0
logging_steps: 10

# preprocessing_num_workers: 1
"""

(LOCAL_CFG_DIR / "config_stage1_pretrain.yaml").write_text(pretrain_yaml)
(LOCAL_CFG_DIR / "config_stage2_sft.yaml").write_text(sft_yaml)

print("Configs written:\n -", LOCAL_CFG_DIR / "config_stage1_pretrain.yaml", "\n -", LOCAL_CFG_DIR / "config_stage2_sft.yaml")


Configs written:
 - /content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml 
 - /content/ft_vs_rag_work/axolotl_configs/config_stage2_sft.yaml


# Cell 6 — (Optional) Preprocess datasets

In [ ]:
# delete cache from previous trials
import shutil, os
#for d in ["cache_pretrain", "cache_sft"]:
for d in ["cache_pretrain"]:
    p = (LOCAL_ROOT / d)
    if p.exists():
        print("Removing", p)
        shutil.rmtree(p)
print("OK")


OK


In [ ]:
#!python -m axolotl.cli.preprocess "/content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml"
#!axolotl preprocess /content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml \
#  --debug-num-examples 0
!python -m axolotl.cli.preprocess "/content/ft_vs_rag_work/axolotl_configs/config_stage2_sft.yaml"


2025-11-13 22:39:02.601316: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 22:39:02.619669: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763073542.641108    7537 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763073542.647707    7537 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763073542.664197    7537 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Check first few lines of datasets to ensure schemas are right
!head -n 2 /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl
!head -n 2 /content/ft_vs_rag_work/data/docs_qa_with_context_ALPACA.jsonl


{"text": "Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).  It plays Hindi, English and regional songs.  It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.  Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.  The Radio station currently plays a mix of Hindi and Regional music.  Abraham Thomas is the CEO of the company. Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).  It pla

# Cell 7 — Train Stage-1 (live log)

In [ ]:
!axolotl train /content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml \
  --debug-num-examples 0


2025-11-13 22:40:08.101431: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 22:40:08.119052: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763073608.140883    8108 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763073608.147521    8108 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763073608.163750    8108 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Live logs; after finish we’ll zip stage dir and copy to Drive
#(LOCAL_LOG_DIR).mkdir(parents=True, exist_ok=True)
#!python -m axolotl.cli.train "/content/ft_vs_rag_work/axolotl_configs/config_stage1_pretrain.yaml" \
#    2>&1 | tee "/content/ft_vs_rag_work/logs/stage1_pretrain.log"


#Cell 8 — Save Stage-1 artifacts to Drive

In [ ]:
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

stage1_zip = LOCAL_ROOT / "mistral_stage1.zip"
if (STAGE1_DIR.exists()) and any(STAGE1_DIR.iterdir()):
    zip_dir(STAGE1_DIR, stage1_zip)
    safe_copy_file(stage1_zip, DRIVE_OUT_DIR / stage1_zip.name)
    print("Backed up Stage-1 adapter to Drive:", DRIVE_OUT_DIR / stage1_zip.name)
else:
    print("WARNING: Stage-1 output dir is empty or missing:", STAGE1_DIR)

# Save log
safe_copy_file(LOCAL_LOG_DIR / "stage1_pretrain.log", DRIVE_OUT_DIR / "stage1_pretrain.log")
print("Copied log:", DRIVE_OUT_DIR / "stage1_pretrain.log")


Backed up Stage-1 adapter to Drive: /content/drive/MyDrive/ft_vs_rag_project/outputs/mistral_stage1.zip


FileNotFoundError: [Errno 2] No such file or directory: '/content/ft_vs_rag_work/logs/stage1_pretrain.log'

#Cell 9 — Train Stage-2 (live log)

In [ ]:
!python -m axolotl.cli.train "/content/ft_vs_rag_work/axolotl_configs/config_stage2_sft.yaml" \
    2>&1 | tee "/content/ft_vs_rag_work/logs/stage2_sft.log"


2025-11-13 21:56:52.706163: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 21:56:52.723898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763071012.744985   19973 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763071012.751426   19973 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763071012.767629   19973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

#Cell 10 — Save Stage-2 artifacts to Drive

In [ ]:
stage2_zip = LOCAL_ROOT / "mistral_stage2.zip"
if (STAGE2_DIR.exists()) and any(STAGE2_DIR.iterdir()):
    zip_dir(STAGE2_DIR, stage2_zip)
    safe_copy_file(stage2_zip, DRIVE_OUT_DIR / stage2_zip.name)
    print("Backed up Stage-2 adapter to Drive:", DRIVE_OUT_DIR / stage2_zip.name)
else:
    print("WARNING: Stage-2 output dir is empty or missing:", STAGE2_DIR)

# Save log
safe_copy_file(LOCAL_LOG_DIR / "stage2_sft.log", DRIVE_OUT_DIR / "stage2_sft.log")
print("Copied log:", DRIVE_OUT_DIR / "stage2_sft.log")


Backed up Stage-2 adapter to Drive: /content/drive/MyDrive/ft_vs_rag_project/outputs/mistral_stage2.zip
Copied log: /content/drive/MyDrive/ft_vs_rag_project/outputs/stage2_sft.log


Cell 11 — Quick sanity inference (LoRA adapter from Stage-2)

In [ ]:
# ==== פרמטרים שאת משנה לפי הסביבה שלך ====
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # או מה שבחרת
ADAPTER_PATH  = "/content/ft_vs_rag_work/outputs/stage1"  # אם יש LoRA; אם אין, השאירי None
USE_4BIT      = False  # True אם את רוצה מצב B (ללא offload). אחרת מצב A.
OFFLOAD_DIR   = "/content/offload"  # מצב A בלבד
DTYPE         = "bfloat16"  # "float16" אם ה-GPU לא תומך BF16 (למשל T4)

# ==== התקנות נדרשות בסביבת קולאב/דוקר (אם עוד לא מותקן) ====
# !pip install -q transformers accelerate bitsandbytes peft

import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import PeftModel

# יצירת תיקיית offload אם נשתמש במצב A
if not USE_4BIT:
    os.makedirs(OFFLOAD_DIR, exist_ok=True)

# בחירת dtype
dtype = torch.bfloat16 if DTYPE == "bfloat16" and torch.cuda.is_bf16_supported() else torch.float16

# טוענים tokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token  # להבטיח padding חוקי

# ===== מצב A: device_map="auto" עם offload לדיסק =====
if not USE_4BIT:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        offload_folder=OFFLOAD_DIR,     # <-- זה פותר את ה-ValueError
        torch_dtype=dtype,
    )

# ===== מצב B: טעינה ב-4bit (לרוב מונעת צורך ב-disk) =====
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        torch_dtype=dtype,
    )

# אם יש לך LoRA/PEFT – טעני את האדפטור מעל הבסיס:
if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    # מעדכנים את המודל עם האדפטור. נשארים עם אותו device_map/offload כפי שהוטען לבסיס.
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()

# ===== sanity check קצר =====
prompt = "Write a short paragraph on the advantages of RAG over Fine-Tuning."
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tok.eos_token_id,
    )

print(tok.decode(out_ids[0], skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Write a short paragraph on the advantages of RAG over Fine-Tuning.

RAG (ReActing Agents for Generative Tasks) has several advantages over fine-tuning when it comes to generating text or performing other generative tasks. For one, RAG is more flexible and adaptable than fine-tuning. With RAG, you don't need to have a large pre-trained model or a specific domain-specific dataset to generate text. Instead, RAG can learn from a few examples and generate text that is coherent and relevant to the input prompt. This makes RAG a more accessible and cost-effective option for generating text, particularly for smaller organizations or individuals


# Training Files preparation

In [6]:
from pathlib import Path
import json, re, hashlib, tempfile, shutil
from typing import Iterable, Union, List, Optional

# ================== נתיבים ==================
# הנתיב הראשי שבו את עובדת (למשל בקולאב)
MAIN_DATA_DIR = Path("/content/ft_vs_rag_work/data")
MAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# מראות (כדי להתאים ללוגים של Axolotl)
MIRROR_DIRS = [Path("/content/ft_vs_rag_work/runpod/data")]
for d in MIRROR_DIRS:
    d.mkdir(parents=True, exist_ok=True)

# מקורות
SRC_STABLE = MAIN_DATA_DIR / "stable_docs.jsonl"               # אופציונלי לשלב הפרה-טריין
SRC_QA     = MAIN_DATA_DIR / "docs_qa_with_context.jsonl"      # חובה ל-SFT

# יעדים (נכתבים ל-MAIN + מועתקים למראות)
DST_PRE = MAIN_DATA_DIR / "docs_for_pretrain.jsonl"
DST_ALP = MAIN_DATA_DIR / "docs_qa_with_context_ALPACA.jsonl"

# ================== פרמטרים ==================
MIN_CHARS_PRETRAIN = 80
MAX_CHARS_PRETRAIN = None
INCLUDE_QA_CONTEXT_IN_PRETRAIN = True

# ================== ניקוי טקסט ==================
CTRL = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")  # משאיר \t \n \r
ZW   = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060]")      # Zero-width וכד'
WS   = re.compile(r"[ \t\f\v]+")
NL3  = re.compile(r"\n{3,}")

def clean_text(t: Union[str, list, dict, None]) -> str:
    # הפיכה למחרוזת נקייה; לא מכניסים כאן מירכאות ידנית—json.dumps יטפל בזה
    if t is None:
        s = ""
    elif isinstance(t, str):
        s = t
    elif isinstance(t, (list, tuple)):
        s = "\n---\n".join(clean_text(x) for x in t if x is not None)
    else:
        s = json.dumps(t, ensure_ascii=False)

    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = CTRL.sub("", s)
    s = ZW.sub("", s)
    s = NL3.sub("\n\n", s)
    s = "\n".join(WS.sub(" ", ln).strip() for ln in s.split("\n"))
    return s.strip()

# ================== IO בטוח ==================
def iter_jsonl(path: Path) -> Iterable[dict]:
    """
    קורא קובץ JSONL מקור. שורות לא תקינות מדולגות (מדווח למסך).
    """
    if not path.exists():
        return
    bad = 0
    with path.open("r", encoding="utf-8-sig", errors="strict") as f:
        for i, line in enumerate(f, 1):
            s = line.strip()
            if not s:
                continue
            try:
                yield json.loads(s)
            except json.JSONDecodeError as e:
                bad += 1
                if bad <= 5:
                    print(f"❌ {path.name}: bad JSON at line {i}: {e} | snippet: {s[:180]}")
                continue
    if bad:
        print(f"⚠️  {path.name}: skipped {bad} malformed lines")

def validate_jsonl(p: Path, max_reports=20) -> list[int]:
    """
    מאמת שכל שורה בקובץ JSONL ניתנת ל-json.loads.
    מחזיר רשימת מספרי שורות בעייתיות (אם יש).
    """
    bad = []
    with p.open("r", encoding="utf-8", errors="strict") as f:  # בלי BOM, בלי 'ignore'
        for i, line in enumerate(f, 1):
            s = line.rstrip("\n")
            if not s:
                continue
            try:
                json.loads(s)
            except Exception:
                bad.append(i)
                if len(bad) >= max_reports:
                    break
    if not bad:
        print(f"✔️  {p.name} validated OK")
    else:
        print(f"❌ {p.name} has {len(bad)} invalid lines; first: {bad[:5]}")
    return bad

def _atomic_write_one(dst: Path, rows: Iterable[dict]) -> int:
    """
    כותב זמנית ל-tmp, מאמת, ואז מחליף את היעד.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with tempfile.NamedTemporaryFile("w", delete=False, dir=str(dst.parent), encoding="utf-8", newline="\n") as tmp:
        tmp_path = Path(tmp.name)
        for r in rows:
            tmp.write(json.dumps(r, ensure_ascii=False) + "\n")
            n += 1
    # ולידציה חזקה לפני החלפה
    bad = validate_jsonl(tmp_path, max_reports=5)
    if bad:
        try:
            tmp_path.unlink()
        finally:
            raise RuntimeError(f"Validation failed for {dst} (bad lines: {bad[:5]})")
    tmp_path.replace(dst)
    return n

def atomic_write_jsonl_with_mirrors(dst: Path, rows: Iterable[dict], mirrors: list[Path]) -> None:
    """
    כותב ליעד הראשי + מעתיק לכל מראה. מאמת אחרי כל כתיבה/העתקה.
    """
    # צריך לצרוך את rows פעמיים → ממירים לרשימה
    buf = list(rows)
    n = _atomic_write_one(dst, buf)
    print(f"✅ wrote {dst} rows={n}")

    # מראות
    for mdir in mirrors:
        mdir.mkdir(parents=True, exist_ok=True)
        mirror_path = mdir / dst.name
        shutil.copyfile(dst, mirror_path)
        # ולידציה גם על ההעתק
        bad = validate_jsonl(mirror_path, max_reports=5)
        if bad:
            raise RuntimeError(f"Mirror validation failed for {mirror_path} (bad lines: {bad[:5]})")
        print(f"🔁 mirrored → {mirror_path}")

# ================== PRETRAIN ==================
CANDIDATE_KEYS = ["text", "content", "body", "raw_text", "page_text"]

def extract_text_from_stable(rec: dict) -> str:
    title = clean_text(rec.get("title", ""))
    txt = ""
    for k in CANDIDATE_KEYS:
        if rec.get(k):
            txt = rec[k]
            break
    txt = clean_text(txt)
    if title:
        if txt and not txt.startswith(title[:50]):
            txt = f"{title}\n\n{txt}"
        elif not txt:
            txt = title
    return txt

def build_pretrain(dst: Path, stable: Optional[Path], qa: Optional[Path]) -> None:
    dedup = set()

    def gen_rows():
        # 1) stable_docs (אם קיים)
        if stable and stable.exists():
            for r in iter_jsonl(stable):
                t = extract_text_from_stable(r)
                if not t:
                    continue
                if len(t) < MIN_CHARS_PRETRAIN:
                    continue
                if MAX_CHARS_PRETRAIN and len(t) > MAX_CHARS_PRETRAIN:
                    continue
                h = hashlib.blake2s(t[:10000].encode("utf-8"), digest_size=16).hexdigest()
                if h in dedup:
                    continue
                dedup.add(h)
                yield {"text": t}

        # 2) אופציונלי: contexts מה-QA
        if INCLUDE_QA_CONTEXT_IN_PRETRAIN and qa and qa.exists():
            for r in iter_jsonl(qa):
                ctx = clean_text(r.get("context", ""))
                if not ctx or len(ctx) < MIN_CHARS_PRETRAIN:
                    continue
                if MAX_CHARS_PRETRAIN and len(ctx) > MAX_CHARS_PRETRAIN:
                    continue
                h = hashlib.blake2s(ctx[:10000].encode("utf-8"), digest_size=16).hexdigest()
                if h in dedup:
                    continue
                dedup.add(h)
                yield {"text": ctx}

    atomic_write_jsonl_with_mirrors(dst, gen_rows(), MIRROR_DIRS)

# ================== ALPACA (SFT) ==================
def build_alpaca(dst: Path, qa: Path) -> None:
    def gen_rows():
        for r in iter_jsonl(qa):
            q   = clean_text(r.get("question", ""))
            a   = clean_text(r.get("answer",   ""))
            ctx = clean_text(r.get("context",  ""))
            if not q or not a:
                continue
            instruction = "Answer the question. Use the provided context if it helps."
            inp = f"<CONTEXT>\n{ctx}\n\nQuestion: {q}" if ctx else f"Question: {q}"
            row = {"instruction": instruction, "input": inp, "output": a}
            if "id" in r:
                row["id"] = r["id"]
            yield row

    atomic_write_jsonl_with_mirrors(dst, gen_rows(), MIRROR_DIRS)

# ================== הרצה ==================
if not SRC_STABLE.exists() and not SRC_QA.exists():
    raise SystemExit("לא נמצאו קלטים: stable_docs.jsonl ו/או docs_qa_with_context.jsonl תחת /content/ft_vs_rag_work/data")

build_pretrain(
    DST_PRE,
    SRC_STABLE if SRC_STABLE.exists() else None,
    None , #SRC_QA if SRC_QA.exists()     else None,
)

if SRC_QA.exists():
    build_alpaca(DST_ALP, SRC_QA)
else:
    print("ℹ️ אין docs_qa_with_context.jsonl – דילוג על יצירת קובץ Alpaca.")

    #TO-USE


✔️  tmp64xg5_ca validated OK
✅ wrote /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl rows=530473
✔️  docs_for_pretrain.jsonl validated OK
🔁 mirrored → /content/ft_vs_rag_work/runpod/data/docs_for_pretrain.jsonl
✔️  tmplxk2d5l7 validated OK
✅ wrote /content/ft_vs_rag_work/data/docs_qa_with_context_ALPACA.jsonl rows=90447
✔️  docs_qa_with_context_ALPACA.jsonl validated OK
🔁 mirrored → /content/ft_vs_rag_work/runpod/data/docs_qa_with_context_ALPACA.jsonl


In [12]:
import json
import tqdm

input_file = '/content/ft_vs_rag_work/data/docs_for_pretrain.jsonl'
output_file = '/content/ft_vs_rag_work/data/docs_for_pretrain_v1.jsonl'
bad_rows = 0
total_rows = 0

with open(input_file, 'r', encoding='utf-8') as infile, \
     open(output_file, 'w', encoding='utf-8') as outfile:

    # עובדים על כל שורה בנפרד
    for line in tqdm.tqdm(infile, desc="Filtering bad JSON lines"):
        total_rows += 1
        try:
            # ניסיון לנתח את השורה
            obj = json.loads(line)
            # אם הניתוח הצליח, כותבים לקובץ החדש
            outfile.write(json.dumps(obj, ensure_ascii=False) + '\n')
        except json.JSONDecodeError:
            # אם הניתוח נכשל, מתעלמים מהשורה
            bad_rows += 1
            pass

print(f"Total rows processed: {total_rows}")
print(f"Bad (ignored) rows: {bad_rows}")
print(f"Fixed dataset saved to: {output_file}")

Filtering bad JSON lines: 530473it [00:24, 21326.45it/s]

Total rows processed: 530473
Bad (ignored) rows: 0
Fixed dataset saved to: /content/ft_vs_rag_work/data/docs_for_pretrain_v1.jsonl


In [15]:
import json
import os
import math
from tqdm import tqdm

# --- הגדרות ---
INPUT_FILE = '/content/ft_vs_rag_work/data/docs_for_pretrain.jsonl'
OUTPUT_BASE_PATH = '/content/ft_vs_rag_work/data/'
NUM_SHARDS = 3

# --- שלב 1: טעינת כל השורות ---
print(f"Loading data from {INPUT_FILE}...")
lines = []
try:
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        # טוען את השורות כדי לחשב את סה"כ הכמות
        lines = f.readlines()
except FileNotFoundError:
    print(f"Error: Input file not found at {INPUT_FILE}")
    exit()

TOTAL_LINES = len(lines)
if TOTAL_LINES == 0:
    print("Error: Input file is empty.")
    exit()

print(f"Total lines found: {TOTAL_LINES}")

# --- שלב 2: חישוב גודל כל שארד ---
# שימוש ב-math.ceil כדי לוודא שכל השורות נכללות, גם אם יש שארית.
SHARD_SIZE = math.ceil(TOTAL_LINES / NUM_SHARDS)
print(f"Lines per shard (approx.): {SHARD_SIZE}")

# --- שלב 3: חלוקה ושמירה לקבצים חדשים ---
start_index = 0

for i in range(NUM_SHARDS):
    # הגדרת שם קובץ חדש: docs_shard_0.jsonl, docs_shard_1.jsonl, וכו'.
    output_filename = os.path.join(OUTPUT_BASE_PATH, f"docs_shard_{i}.jsonl")

    # חישוב טווח השורות לשארד הנוכחי
    end_index = min(start_index + SHARD_SIZE, TOTAL_LINES)
    shard_lines = lines[start_index:end_index]

    # כתיבת השורות לקובץ
    with open(output_filename, 'w', encoding='utf-8') as outfile:
        outfile.writelines(shard_lines)

    print(f"✅ Saved Shard {i} to {output_filename} with {len(shard_lines)} lines.")

    # עדכון אינדקס ההתחלה לשארד הבא
    start_index = end_index

print("\nAll files successfully sharded.")

Loading data from /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl...
Total lines found: 530473
Lines per shard (approx.): 176825
✅ Saved Shard 0 to /content/ft_vs_rag_work/data/docs_shard_0.jsonl with 176825 lines.
✅ Saved Shard 1 to /content/ft_vs_rag_work/data/docs_shard_1.jsonl with 176825 lines.
✅ Saved Shard 2 to /content/ft_vs_rag_work/data/docs_shard_2.jsonl with 176823 lines.

All files successfully sharded.
